In [ ]:
print("sasa")

sasa


In [8]:
from langchain_groq import ChatGroq



from dotenv import load_dotenv
load_dotenv()




True

In [9]:
model = ChatGroq(model="qwen/qwen3-32b")

model.invoke("Hello, how are you?").content

'<think>\nOkay, the user asked "Hello, how are you?" so I need to respond politely. I should acknowledge their greeting and offer a friendly reply. Maybe start with a cheerful "Hi!" to set a positive tone. Then mention that I\'m functioning well to reassure them. It\'s important to ask how they\'re doing to keep the conversation two-way. I should keep it concise but warm, avoiding any technical jargon. Let me make sure the response is inviting and encourages them to share more. Yeah, that should work.\n</think>\n\nHi there! 😊 I\'m doing well, thank you! How can I assist you today?'

In [4]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings

/Users/harshsaw/Github/document_RAG/env/lib/python3.10/site-packages/google/api_core/_python_version_support.py:266: FutureWarning: You are using a Python version (3.10.19) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)


In [5]:
embedding_model =GoogleGenerativeAIEmbeddings(model="gemini-embedding-001")

[-0.03931594267487526,
 0.012321420013904572,
 0.0014646905474364758,
 -0.08070500195026398,
 -0.02218940295279026,
 0.002888158429414034,
 -0.022148549556732178,
 0.006302330177277327,
 0.006308619864284992,
 0.0021026867907494307,
 -0.009357460774481297,
 -0.007212435360997915,
 -0.004258986096829176,
 0.0029940365348011255,
 0.16378968954086304,
 -0.014995668083429337,
 0.005600892938673496,
 0.007743187248706818,
 -0.0038035246543586254,
 -0.017707178369164467,
 -0.0062597040086984634,
 0.005769195966422558,
 -0.0033910837955772877,
 -0.0011192664969712496,
 -0.019598782062530518,
 0.003407731419429183,
 0.01065141148865223,
 -0.004336030688136816,
 0.022021060809493065,
 0.010780487209558487,
 0.0027617833111435175,
 -0.006386036053299904,
 -0.011868474073708057,
 0.014990522526204586,
 0.00893055647611618,
 0.005228688474744558,
 0.011220615357160568,
 -0.005573965609073639,
 -0.009834540076553822,
 -9.500277519691736e-05,
 0.01163129135966301,
 0.0011866700369864702,
 -0.0065536

In [14]:
##data ingestion

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [15]:
import os 
file_path = os.path.join(os.getcwd() ,"data", "sample.pdf")

loader = PyPDFLoader(file_path=file_path)
documents = loader.load()

In [ ]:
len(documents)

77

In [16]:
#chunking the data

text_splitter = RecursiveCharacterTextSplitter(
    
    chunk_size=250,
    chunk_overlap=50,
    length_function=len
)

In [17]:
docs = text_splitter.split_documents(documents)

In [18]:
len(docs)

1329

SyntaxError: invalid syntax (509059171.py, line 1)

In [19]:
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings

# Easiest local embedding model - no API key needed
embedding_model = HuggingFaceEmbeddings(
    model_name="all-MiniLM-L6-v2"  # Fast, lightweight, good quality
)

# Create vectorstore
vectorstore = FAISS.from_documents(docs, embedding_model)

/var/folders/5s/2n82n8212_1d3prjyg6h1xl00000gn/T/ipykernel_3026/1356997058.py:5: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embedding_model = HuggingFaceEmbeddings(


In [ ]:
retrieved = vectorstore.similarity_search("What is LangChain?", k=2)


In [20]:
retriever = vectorstore.as_retriever(k=10)

In [ ]:
retriever.invoke("What is LangChain?")

[Document(id='17ed2edc-5ba4-4745-a7c8-362033474b87', metadata={'producer': 'pdfTeX-1.40.25', 'creator': 'LaTeX with hyperref', 'creationdate': '2023-07-20T00:30:36+00:00', 'author': '', 'keywords': '', 'moddate': '2023-07-20T00:30:36+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5', 'subject': '', 'title': '', 'trapped': '/False', 'source': '/Users/harshsaw/Github/document_RAG/notebook/data/sample.pdf', 'total_pages': 77, 'page': 37, 'page_label': '38'}, page_content='Language Processing, pages 2174–2184, 2018.\nAakanksha Chowdhery, Sharan Narang, Jacob Devlin, Maarten Bosma, Gaurav Mishra, Adam Roberts,\nPaul Barham, Hyung Won Chung, Charles Sutton, Sebastian Gehrmann, Parker Schuh, Kensen Shi, Sasha'),
 Document(id='10e6f316-d1e1-44d1-bf11-01d5933b2932', metadata={'producer': 'pdfTeX-1.40.25', 'creator': 'LaTeX with hyperref', 'creationdate': '2023-07-20T00:30:36+00:00', 'author': '', 'keywords': '', 'moddate': '2023-

In [21]:
from langchain_core.prompts import PromptTemplate

prompts = PromptTemplate(
    input_variables=["context", "question"],
    template="Use the following context to answer the question . If context does not help, say 'I don't know' . Context: {context} Question: {question}\n\n Answer:"
)   

In [22]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough


In [ ]:
parser = StrOutputParser()

In [23]:
def format_docs(docs):
    return "\n".join([doc.page_content for doc in docs])

In [24]:
rag_chain=(
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompts | model | StrOutputParser( )
    
    )

In [25]:
rag_chain.invoke("What is llama2 finetuning benchmark experiments")

"<think>\nOkay, let's see. The user is asking about the finetuning benchmark experiments for Llama 2. I need to check the provided context for any information on that.\n\nLooking at the context, the first part mentions that testing conducted to date has been in English and hasn't covered certain aspects. Then in the Training Data section, it says that Llama 2 was pre-trained on 2 trillion tokens from public sources, and fine-tuning used publicly available instruction datasets. It also notes that like other LLMs, Llama 2 might generate harmful content, and they tried to mitigate this with fine-tuning, but some issues might remain.\n\nWait, the context doesn't specifically mention any finetuning benchmark experiments. It talks about the data used for pretraining and fine-tuning, and some mitigation efforts. But there's no explicit information about the benchmarks or experiments conducted during finetuning. The user is asking about the benchmarks, so unless there's a part I missed, the co